In [39]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [40]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: IT_Security_Manual.pdf
  ✓ Loaded 3 pages

Processing: HR_Policy_Document.pdf
  ✓ Loaded 3 pages

Processing: University_Handbook.pdf
  ✓ Loaded 3 pages

Processing: Banking_Policy_Document.pdf
  ✓ Loaded 3 pages

Total documents loaded: 12


In [41]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/IT_Security_Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'IT_Security_Manual.pdf', 'file_type': 'pdf'}, page_content='IT Security and Compliance Manual\nPassword Policy\nPasswords must be at least 12 characters long.\nPasswords must include uppercase, lowercase, numbers, and symbols.\nPasswords must be changed every 90 days.'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source'

In [42]:
### Split documents into chunks using RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],
)

pdf_chunks = text_splitter.split_documents(all_pdf_documents)
print(f"Split {len(all_pdf_documents)} documents into {len(pdf_chunks)} chunks")

Split 12 documents into 12 chunks


In [43]:
chunks = text_splitter.split_documents(all_pdf_documents)
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/IT_Security_Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'IT_Security_Manual.pdf', 'file_type': 'pdf'}, page_content='IT Security and Compliance Manual\nPassword Policy\nPasswords must be at least 12 characters long.\nPasswords must include uppercase, lowercase, numbers, and symbols.\nPasswords must be changed every 90 days.'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source'

Embedding and vector DB


In [44]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [45]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1800.42it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


Vector store

Vector Store

In [46]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 36


In [47]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/IT_Security_Manual.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'IT_Security_Manual.pdf', 'file_type': 'pdf'}, page_content='IT Security and Compliance Manual\nPassword Policy\nPasswords must be at least 12 characters long.\nPasswords must include uppercase, lowercase, numbers, and symbols.\nPasswords must be changed every 90 days.'),
 Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-18T14:45:49+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-18T14:45:49+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source'

In [48]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 12 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.20it/s]

Generated embeddings with shape: (12, 384)
Adding 12 documents to vector store...
Successfully added 12 documents to vector store
Total documents in collection: 48


In [49]:
vectorstore

Retriever Pipeline From VectorStore

In [50]:
from __future__ import annotations
from typing import List, Dict, Any

class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Chroma uses L2 by default; distance is typically in [0, ~2] for normalized embeddings.
                    # Map to similarity in [0, 1]: 1 = best match, 0 = worst.
                    similarity_score = max(0.0, 1.0 - (distance / 2.0))
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

# Create retriever only when dependencies exist (run EmbeddingManager and VectorStore cells first)
if "vectorstore" in globals() and "embedding_manager" in globals():
    rag_retriever = RAGRetriever(vectorstore, embedding_manager)
else:
    rag_retriever = None  # Run the VectorStore and EmbeddingManager cells first, then re-run this cell

In [51]:
rag_retriever

In [52]:
# Ensure retriever exists if dependencies are available
if rag_retriever is None and "vectorstore" in globals() and "embedding_manager" in globals():
    rag_retriever = RAGRetriever(vectorstore, embedding_manager)

results = rag_retriever.retrieve("Sick leave is limited to how many days", top_k=3, score_threshold=0.0)
print(f"\n--- {len(results)} result(s) ---")
for i, doc in enumerate(results, 1):
    print(f"\n[{i}] score={doc['similarity_score']:.3f} | {doc['metadata'].get('source_file', '')}")
    print(doc["content"])
results

Retrieving documents for query: 'Sick leave is limited to how many days'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 82.89it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)

--- 3 result(s) ---

[1] score=0.725 | HR_Policy_Document.pdf
Company HR Policy Manual
Leave Policy
Employees are entitled to 20 paid vacation days per year.
Sick leave is limited to 10 days annually.
Unused vacation days can be carried forward up to 5 days.
Maternity leave is provided for 26 weeks.

[2] score=0.725 | HR_Policy_Document.pdf
Company HR Policy Manual
Leave Policy
Employees are entitled to 20 paid vacation days per year.
Sick leave is limited to 10 days annually.
Unused vacation days can be carried forward up to 5 days.
Maternity leave is provided for 26 weeks.

[3] score=0.725 | HR_Policy_Document.pdf
Company HR Policy Manual
Leave Policy
Employees are entitled to 20 paid vacation days per year.
Sick leave is limited to 10 days annually.
Unused vacation days can be carried forward up to 5 days.
Maternity leave is provided for 26 weeks.


[{'id': 'doc_a298fcf2_3',
  'content': 'Company HR Policy Manual\nLeave Policy\nEmployees are entitled to 20 paid vacation days per year.\nSick leave is limited to 10 days annually.\nUnused vacation days can be carried forward up to 5 days.\nMaternity leave is provided for 26 weeks.',
  'metadata': {'subject': '(unspecified)',
   'creationdate': '2026-02-18T14:45:49+00:00',
   'keywords': '',
   'doc_index': 3,
   'page_label': '1',
   'page': 0,
   'content_length': 238,
   'file_type': 'pdf',
   'moddate': '2026-02-18T14:45:49+00:00',
   'trapped': '/False',
   'source': '../data/pdf/HR_Policy_Document.pdf',
   'creator': '(unspecified)',
   'author': '(anonymous)',
   'producer': 'ReportLab PDF Library - www.reportlab.com',
   'total_pages': 3,
   'source_file': 'HR_Policy_Document.pdf',
   'title': '(anonymous)'},
  'similarity_score': 0.7247256338596344,
  'distance': 0.5505487322807312,
  'rank': 1},
 {'id': 'doc_0806a4d5_3',
  'content': 'Company HR Policy Manual\nLeave Policy\n